In [1]:
import sys
if '/home/jovyan/src' not in sys.path:
    sys.path.append('/home/jovyan/src')

from spark_utils import crear_spark_session
from gold.predictivo import construir_features_clasificacion

In [2]:
spark = crear_spark_session()
construir_features_clasificacion(spark)
spark.stop()

2026-07-17 14:57:56,256 [INFO] SparkSession creada: 3.5.0
2026-07-17 14:57:56,257 [INFO] Iniciando construcción de features para clasificación (Aeropuerto)...
2026-07-17 14:58:11,494 [INFO] Gold escrito (overwrite): predictivo/features_clasificacion (373,810 filas)
2026-07-17 14:58:11,516 [INFO]   features_clasificacion: yellow 2023 listo
2026-07-17 14:58:18,813 [INFO] Gold escrito (append): predictivo/features_clasificacion (400,475 filas)
2026-07-17 14:58:18,821 [INFO]   features_clasificacion: yellow 2024 listo
2026-07-17 14:58:24,730 [INFO] Gold escrito (append): predictivo/features_clasificacion (446,552 filas)
2026-07-17 14:58:24,736 [INFO]   features_clasificacion: yellow 2025 listo
2026-07-17 14:58:25,685 [INFO] Gold escrito (append): predictivo/features_clasificacion (7,384 filas)
2026-07-17 14:58:25,689 [INFO]   features_clasificacion: green 2023 listo
2026-07-17 14:58:26,490 [INFO] Gold escrito (append): predictivo/features_clasificacion (6,159 filas)
2026-07-17 14:58:26,493

In [1]:
import sys
if '/home/jovyan/src' not in sys.path:
    sys.path.append('/home/jovyan/src')

from config import GOLD_PATH
from spark_utils import crear_spark_session

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# Crear sesión de Spark para leer los datos
spark = crear_spark_session()

# 1. Cargar el dataset muestreado desde la capa Gold
df_class_spark = spark.read.parquet(str(GOLD_PATH / "predictivo" / "features_clasificacion"))
df_class_pd = df_class_spark.toPandas()

# 2. Separar las variables predictoras (X) y la variable objetivo (y)
# Quitamos 'target_aeropuerto' porque es lo que queremos predecir, y 'anio' porque no es predictivo
X = df_class_pd.drop(columns=["target_aeropuerto", "anio"])
y = df_class_pd["target_aeropuerto"]

# 3. Dividir los datos: 70% para entrenar, 30% para evaluar (Test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

print(f"Filas para entrenamiento: {len(X_train)}")
print(f"Filas para prueba: {len(X_test)}")

# 4. Instanciar y entrenar el modelo Random Forest ajustado
rf_model_bal = RandomForestClassifier(
    n_estimators=100, 
    max_depth=15,             # Aumentamos la profundidad para encontrar patrones más finos
    class_weight='balanced',  # Penaliza severamente fallar en la predicción de la clase 1
    random_state=42, 
    n_jobs=-1
)
rf_model_bal.fit(X_train, y_train)

# 5. Hacer predicciones sobre los datos de prueba
y_pred_bal = rf_model_bal.predict(X_test)

# 6. Evaluar el rendimiento del nuevo modelo
print("\n--- Nuevo Reporte de Clasificación (Balanceado) ---")
print(classification_report(y_test, y_pred_bal))

# Liberar memoria de Spark
spark.stop()

2026-07-17 16:00:15,985 [INFO] NumExpr defaulting to 4 threads.
2026-07-17 16:00:24,393 [INFO] SparkSession creada: 3.5.0


Filas para entrenamiento: 5859254
Filas para prueba: 2511110

--- Nuevo Reporte de Clasificación (Balanceado) ---
              precision    recall  f1-score   support

           0       0.99      0.88      0.93   2393767
           1       0.27      0.90      0.42    117343

    accuracy                           0.88   2511110
   macro avg       0.63      0.89      0.68   2511110
weighted avg       0.96      0.88      0.91   2511110



In [3]:
import pandas as pd

# Crear un DataFrame rápido solo con las respuestas
df_resultados = pd.DataFrame({
    'Real': y_test,
    'Prediccion': y_pred_bal
})

# Reemplazar los 0 y 1 por texto para que Power BI lo entienda directo
df_resultados['Real'] = df_resultados['Real'].map({0: 'Normal', 1: 'Aeropuerto'})
df_resultados['Prediccion'] = df_resultados['Prediccion'].map({0: 'Normal', 1: 'Aeropuerto'})

# Exportar
df_resultados.to_csv('matriz_rf.csv', index=False)